## Model Training

In [1]:
import torch
import torch.optim as optim

In [2]:
# Loss Function - Dice Loss
def dice_loss(predictions, targets, smooth=1e-6):
    # Apply sigmoid to convert logits to 0-1
    predictions = torch.sigmoid(predictions)
    
    # Flatten both tensors
    predictions = predictions.view(-1)
    targets = targets.view(-1)
    
    # Compute intersection and dice score
    intersection = (predictions * targets).sum()
    dice_score = (2 * intersection + smooth) / (predictions.sum() + targets.sum() + smooth)
    
    return 1 - dice_score

In [3]:
import sys
sys.path.append('../src')
from model import UNet

model = UNet(in_channels=1, out_channels=1)

In [4]:
LEARNING_RATE = 1e-4
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [5]:
BATCH_SIZE = 16
EPOCHS = 1
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print(f"Using device: {DEVICE}")
model = UNet(in_channels=1, out_channels=1).to(DEVICE)

Using device: mps


In [6]:
import os
from glob import glob
from dataset import split_datasets

DATA_PATH_IMG = "../Chest-X-Ray/image"
DATA_PATH_MSK = "../Chest-X-Ray/mask"

image_paths = sorted(glob(os.path.join(DATA_PATH_IMG, '[0-9]*')))
mask_paths = sorted(glob(os.path.join(DATA_PATH_MSK, '[0-9]*')))

train_dataset, val_dataset, test_dataset = split_datasets(image_paths, mask_paths)

Found 492 image-mask pairs
Found 106 image-mask pairs
Found 106 image-mask pairs


In [7]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [8]:
from tqdm import tqdm

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_dice = 0.0

    pbar = tqdm(loader, desc='Training')

    for images, masks in pbar:
        images = images.to(device)
        masks = masks.to(device).unsqueeze(1)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        running_dice += loss.item()
        loss.backward()
        optimizer.step()

        pbar.set_postfix({'dice loss': f'{loss.item():.4f}'})

    return running_dice / len(loader)

In [9]:
def validate(model, loader, criterion, device):
    model.eval()
    running_dice = 0.0

    pbar = tqdm(loader, desc='Validation')

    with torch.no_grad():
        for images, masks in pbar:
            images = images.to(device)
            masks = masks.to(device).unsqueeze(1)

            outputs = model(images)
            dice = dice_loss(outputs, masks)
            running_dice += dice.item()
            
            pbar.set_postfix({'loss': f'{dice.item():.4f}'})
    
    return running_dice / len(loader)

In [10]:
train_dices, val_dices = [], []
best_val_dice = float('inf')

for epoch in range(EPOCHS):
        print(f"\nEpoch {epoch+1}/{EPOCHS}")
        
        # Train
        train_dice = train_one_epoch(model, train_loader, dice_loss, optimizer, DEVICE)
        train_dices.append(train_dice)
        
        # Validate
        val_dice = validate(model, val_loader, dice_loss, DEVICE)
        val_dices.append(val_dice)
        
        print(f"Train Dice: {train_dice:.4f}")
        print(f"Val Dice: {val_dice:.4f}")
        
        # Save best model
        if val_dice < best_val_dice:
            best_val_dice = val_dice
            torch.save(model.state_dict(), '../outputs/checkpoints/best_model.pth')
            print(f"Saved best model (Dice: {val_dice:.4f})")
    
print("\n" + "="*50)
print("Training Complete!")
print(f"Best Val Dice Score: {best_val_dice:.4f}")
print("="*50)


Epoch 1/1


Validation: 100%|██████████| 7/7 [05:19<00:00, 45.58s/it, loss=0.6357]


Train Dice: 0.6725
Val Dice: 0.6580


RuntimeError: Parent directory ../outputs/checkpoints does not exist.

In [ ]:
import torch
torch.mps.empty_cache()